<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-5-Lab-2/Unit_5_Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Step 1: Import Required Libraries


In [ ]:
import tensorflow.compat.v1 as tf
tf.disable_eager_execution()
tf.disable_v2_behavior()

from aif360.metrics import ClassificationMetric
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_adult
from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing
from sklearn.preprocessing import MaxAbsScaler
import matplotlib.pyplot as plt

##Step 2: Load and Split the Dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from aif360.datasets import StandardDataset

# Column names from the UCI Adult dataset documentation
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]

# Load raw data from UCI ML repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df = pd.read_csv(url, names=column_names, sep=",\\s*", engine="python")

# Preprocessing: convert 'sex' and 'income' to binary, drop rows with missing values
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
df.dropna(inplace=True)

# Separate features and labels
X = df.drop(['income'], axis=1)
X = pd.get_dummies(X)  # One-hot encode categorical features
y = df['income']

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, shuffle=True, random_state=42)

# Convert to AIF360 StandardDataset format for compatibility with AdversarialDebiasing
from aif360.datasets import BinaryLabelDataset

train_df = X_train.copy()
train_df['income'] = y_train
test_df = X_test.copy()
test_df['income'] = y_test

dataset_adult_train = StandardDataset(train_df,
    label_name='income',
    favorable_classes=[1],
    protected_attribute_names=['sex'],
    privileged_classes=[[1]])

dataset_adult_test = StandardDataset(test_df,
    label_name='income',
    favorable_classes=[1],
    protected_attribute_names=['sex'],
    privileged_classes=[[1]])

##Step 3: Define Privileged vs. Unprivileged Groups

In [ ]:
privileged_groups = [{'sex': 1}]   # Male
unprivileged_groups = [{'sex': 0}] # Female

##Step 4: Normalize Features

In [ ]:
min_max_scaler = MaxAbsScaler()
dataset_adult_train.features = min_max_scaler.fit_transform(dataset_adult_train.features)
dataset_adult_test.features = min_max_scaler.transform(dataset_adult_test.features)

##Step 5: Train Adversarial Debiasing Model

In [ ]:
sess = tf.Session()
debiased_model = AdversarialDebiasing(
    privileged_groups=privileged_groups,
    unprivileged_groups=unprivileged_groups,
    scope_name='debiased_classifier',
    debias=True,
    sess=sess
)
debiased_model.fit(dataset_adult_train)

##Step 6: Predict on Test Set

In [ ]:
dataset_adult_pred = debiased_model.predict(dataset_adult_test)

##Step 7: Define Reusable Evaluation Function

In [ ]:
def evaluate_fairness(name, y_true, y_pred):
    m = ClassificationMetric(
        y_true, y_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )
    print(f"=== {name} ===")
    print("Accuracy:", m.accuracy())
    print("Statistical Parity Difference:", m.statistical_parity_difference())
    print("→ A value closer to 0 means both groups receive positive outcomes at similar rates.")
    print("Equal Opportunity Difference:", m.equal_opportunity_difference())
    print("→ A value closer to 0 means both groups have equal true positive rates.")

##Step 8: Evaluate the Debiased Model

In [ ]:
evaluate_fairness("Debiased Model", dataset_adult_test, dataset_adult_pred)

##Step 9: Train Baseline Model (No Debiasing)

In [ ]:
sess_baseline = tf.Session()
baseline_model = AdversarialDebiasing(
    privileged_groups=privileged_groups,
    unprivileged_groups=unprivileged_groups,
    scope_name='baseline_classifier',
    debias=False,
    sess=sess_baseline
)
baseline_model.fit(dataset_adult_train)
baseline_pred = baseline_model.predict(dataset_adult_test)

##Step 10: Evaluate Baseline Model

In [ ]:
evaluate_fairness("Baseline Model", dataset_adult_test, baseline_pred)

##Step 11: Visualize Group Prediction Distributions

In [ ]:
import matplotlib.pyplot as plt

def plot_group_distribution(dataset, title="Label Distribution by Group"):
    """
    Plots the distribution of predicted or actual labels
    across privileged and unprivileged groups.
    """
    labels = dataset.labels.ravel()
    groups = dataset.protected_attributes.ravel()

    # Split label counts by group (e.g., male vs female)
    privileged_labels = labels[groups == 1]
    unprivileged_labels = labels[groups == 0]

    # Create histogram side-by-side
    plt.hist(
        [privileged_labels, unprivileged_labels],
        label=["Privileged (Male)", "Unprivileged (Female)"],
        bins=2, align='left', rwidth=0.8
    )

    # Axis settings
    plt.xticks([0, 1], ["Negative", "Positive"])
    plt.xlabel("Outcome Label")
    plt.ylabel("Count")
    plt.title(title)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.show()

# Call for actual and predicted distributions
plot_group_distribution(dataset_adult_test, "Actual Label Distribution by Group")
plot_group_distribution(dataset_adult_pred, "Predicted Label Distribution by Group (Debiased)")

### 🔍 How to Read These Plots

- **Bars show how many people in each group (male/female) received each outcome**.
- If the model is fair, we expect more **balanced positive outcomes** (right bar).
- The actual data is imbalanced — many more men earn >50K.
- The debiased model adjusts for this by giving more positive predictions to women.

> Use these visualizations to connect fairness metrics (like statistical parity) to how the model is treating people differently based on protected attributes.

##Interactive Widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widget controls
debias_toggle = widgets.ToggleButtons(
    options=[True, False],
    description='Debiasing:',
    value=True
)
epoch_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description='Epochs:'
)
adv_weight_slider = widgets.FloatSlider(
    value=0.1, min=0.0, max=1.0, step=0.05,
    description='Adv Weight:'
)

run_button = widgets.Button(description="Train Model")

# Output display
out = widgets.Output()

def on_run_button_clicked(b):
    clear_output(wait=True)
    display(debias_toggle, epoch_slider, adv_weight_slider, run_button, out)
    with out:
        tf.reset_default_graph()
        sess = tf.Session()
        model = AdversarialDebiasing(
            privileged_groups=privileged_groups,
            unprivileged_groups=unprivileged_groups,
            scope_name='interactive_classifier',
            debias=debias_toggle.value,
            sess=sess,
            num_epochs=epoch_slider.value,
            adversary_loss_weight=adv_weight_slider.value
        )
        model.fit(dataset_adult_train)
        pred = model.predict(dataset_adult_test)
        evaluate_fairness("Interactive Model", dataset_adult_test, pred)
        plot_group_distribution(pred, "Prediction Distribution (Interactive)")

run_button.on_click(on_run_button_clicked)

# Display widgets
display(debias_toggle, epoch_slider, adv_weight_slider, run_button, out)

### Interactive Fairness Tuning

Use the widget above to experiment with the **Adversarial Debiasing model**.  
You can control the following parameters:

#### Debiasing (True / False)
- Turns on or off the fairness constraint.
- When **True**, the model actively tries to reduce bias in the representation.
- When **False**, the model acts like a standard classifier with no fairness constraint.

#### Epochs (1–50)
- Controls how many times the model sees the training data.
- Higher values often improve learning, but can also cause overfitting.

#### Adversary Loss Weight (0.0–1.0)
- Controls **how much weight is given to fairness loss** vs classification accuracy.
- A higher value pushes the model to reduce bias more aggressively.
- A lower value lets the model focus more on raw predictive performance.

---

### How to Interpret Results

After training, examine:

1. **Accuracy** – Did the model predict well overall?
2. **Statistical Parity Difference**
   - Near **0** means both groups received positive predictions at similar rates.
   - Positive = favors unprivileged group, negative = favors privileged group.
3. **Equal Opportunity Difference**
   - Near **0** means both groups had equal chance of being correctly labeled as positive (true positives).
   - Large values may indicate hidden performance gaps between groups.

Try toggling debiasing **on and off**. Then adjust `Adv Weight` to see how the fairness metrics respond.
You are training and comparing **fair vs. unfair algorithms** in real time.